# HiT-MAC Training on Google Colab

**Before running:** Runtime → Change runtime type → T4 GPU

In [ ]:
# ── 1. Clone repo ──────────────────────────────────────────────────────────────
import os
if not os.path.exists('DSN_NA2Q'):
    !git clone https://github.com/Chiseled141/DSN_NA2Q.git
%cd DSN_NA2Q

In [ ]:
# ── 2. Mount Google Drive (strongly recommended) ───────────────────────────────
import os
from google.colab import drive
drive.mount('/content/drive')

DRIVE_CKPT_DIR = '/content/drive/MyDrive/DSN_NA2Q/hitmac/checkpoints'
os.makedirs(DRIVE_CKPT_DIR, exist_ok=True)
print(f'Drive mounted → {DRIVE_CKPT_DIR}')

In [ ]:
# ── 3. Install dependencies ────────────────────────────────────────────────────
!pip install -q -r requirements.txt

In [ ]:
# ── 4. Verify GPU ──────────────────────────────────────────────────────────────
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('Memory:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')

In [ ]:
# ── 5. Config ──────────────────────────────────────────────────────────────────
SCENARIO = 1     # 1 = small (5 sensors, 6 targets)
                 # 2 = large (50 sensors, 60 targets)
                 # 3 = medium (15 sensors, 20 targets)
RESUME   = False # True = resume from Drive checkpoint

In [ ]:
# ── 6. Restore checkpoint from Drive (if resuming) ─────────────────────────────
import shutil

LOCAL_CKPT_DIR = 'hitmac/checkpoints'
os.makedirs(LOCAL_CKPT_DIR, exist_ok=True)

if RESUME:
    restored = []
    for fname in os.listdir(DRIVE_CKPT_DIR):
        shutil.copy(os.path.join(DRIVE_CKPT_DIR, fname),
                    os.path.join(LOCAL_CKPT_DIR, fname))
        restored.append(fname)
    if restored:
        print('Restored:', restored)
    else:
        print('Nothing on Drive — starting fresh.')
        RESUME = False

In [ ]:
# ── 7. Train HiT-MAC (runs as subprocess to support multiprocessing) ───────────
# HiT-MAC uses A3C with parallel workers. Running via shell avoids the
# Jupyter multiprocessing limitation where mp.Process workers silently die.

cmd = f'python -m hitmac.main --mode train --scenario {SCENARIO}'
if RESUME:
    cmd += ' --resume'

print('Running:', cmd)
!{cmd}

In [ ]:
# ── 8. Save to Google Drive ────────────────────────────────────────────────────
SAVE_FILES = ['best.pt', 'latest.pt', 'training_history.npz', 'training.log']

saved = []
for fname in SAVE_FILES:
    src = os.path.join(LOCAL_CKPT_DIR, fname)
    if os.path.exists(src):
        shutil.copy(src, os.path.join(DRIVE_CKPT_DIR, fname))
        saved.append(fname)

print('Saved to Drive:', saved)

In [ ]:
# ── 9. Download to your machine ────────────────────────────────────────────────
from google.colab import files

for fname in SAVE_FILES:
    path = os.path.join(LOCAL_CKPT_DIR, fname)
    if os.path.exists(path):
        files.download(path)
    else:
        print(f'Not found: {fname}')